In [1]:
import json
import random
import pickle
import numpy as np

import nltk
from nltk.stem import WordNetLemmatizer

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import SGD

In [2]:
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Aditi.m\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Aditi.m\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Aditi.m\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [3]:
lemmatizer = WordNetLemmatizer()

with open("../data/intents.json") as file:
    intents = json.load(file)

print("Dataset Loaded Successfully")

Dataset Loaded Successfully


In [4]:
words = []
classes = []
documents = []

ignore_letters = ['?', '!', '.', ',']

In [5]:
for intent in intents['intents']:

    for pattern in intent['patterns']:

        word_list = nltk.word_tokenize(pattern)

        words.extend(word_list)

        documents.append((word_list, intent['tag']))

        if intent['tag'] not in classes:
            classes.append(intent['tag'])

In [6]:
words = [
    lemmatizer.lemmatize(word.lower())
    for word in words
    if word not in ignore_letters
]

words = sorted(set(words))
classes = sorted(set(classes))

print("Vocabulary Size :", len(words))
print("Classes :", classes)

Vocabulary Size : 27
Classes : ['face_recognition', 'goodbye', 'greeting', 'product_classification', 'sentiment_analysis', 'thanks']


In [7]:
training = []

output_empty = [0] * len(classes)

In [8]:
for document in documents:

    bag = []

    word_patterns = [
        lemmatizer.lemmatize(word.lower())
        for word in document[0]
    ]

    for word in words:
        bag.append(1 if word in word_patterns else 0)

    output_row = output_empty[:]
    output_row[classes.index(document[1])] = 1

    training.append([bag, output_row])

In [9]:
random.shuffle(training)

training = np.array(training, dtype=object)

train_x = np.array(list(training[:, 0]))
train_y = np.array(list(training[:, 1]))

In [10]:
model = Sequential()

model.add(Dense(128, input_shape=(len(train_x[0]),), activation="relu"))
model.add(Dropout(0.5))

model.add(Dense(64, activation="relu"))
model.add(Dropout(0.5))

model.add(Dense(len(train_y[0]), activation="softmax"))

c:\Users\Aditi.m\smart-retail-ai\venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [11]:
optimizer = SGD(
    learning_rate=0.01,
    momentum=0.9
)

model.compile(
    loss="categorical_crossentropy",
    optimizer=optimizer,
    metrics=["accuracy"]
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         3,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,230 (47.77 KB)

 Trainable params: 12,230 (47.77 KB)

 Non-trainable params: 0 (0.00 B)

In [12]:
history = model.fit(
    train_x,
    train_y,
    epochs=200,
    batch_size=8,
    verbose=1
)

Epoch 1/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 28ms/step - accuracy: 0.1500 - loss: 1.8099  
Epoch 2/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.1500 - loss: 1.8289 
Epoch 3/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.1500 - loss: 1.7726
Epoch 4/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.2500 - loss: 1.7986    
Epoch 5/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.1000 - loss: 1.8096 
Epoch 6/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.4500 - loss: 1.6588 
Epoch 7/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.4500 - loss: 1.7070 
Epoch 8/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.4000 - loss: 1.7017 
Epoch 9/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.3000 - loss: 1.6204
Epoch 10/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.2500 - loss: 1.7101 
Epoch 11/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.3500 - loss: 1.5837
Epoch 12/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy:

In [13]:
model.save("../app/models/chatbot_model.h5")

In [14]:
pickle.dump(words, open("../app/models/chatbot_words.pkl", "wb"))
pickle.dump(classes, open("../app/models/chatbot_classes.pkl", "wb"))

print("Chatbot Saved Successfully")

Chatbot Saved Successfully
